# Indic Transcriber

Transcribe Tamil, Malayalam, Hindi, Kannada, English and mixed speech on Colab's free GPU. Click **Runtime → Run all**, wait a few minutes, then click the link printed at the bottom.

**First time only:**
1. **Runtime → Change runtime type → T4 GPU**.
2. Create a Hugging Face *read* token at https://huggingface.co/settings/tokens, then in Colab click the 🔑 **Secrets** icon on the left, add `HF_TOKEN`, and turn on **Notebook access**.
3. Accept the licenses at https://huggingface.co/bodhan-ai/indic-transcribe-flex and https://huggingface.co/bodhan-ai/indic-transcribe-core (same Hugging Face account as the token).

Built with Indic-Transcribe from Bodhan AI / AI4Bharat.

In [ ]:
# @title Settings
DEFAULT_ENGINE = "indic-flex"  # @param ["indic-flex", "indic-core", "whisper"]
REPO_BRANCH = "main"  # @param {type:"string"}
REPO_URL = "https://github.com/Abhijith2k20/transcribe"

In [ ]:
# @title Install
import os, subprocess, sys

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode:
    print("No GPU found. Go to Runtime → Change runtime type → T4 GPU, then Runtime → Run all again.\n"
          "(It still works without a GPU, just much more slowly.)")
else:
    print(gpu.stdout.strip())

REPO_DIR = "/content/transcribe"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
sys.path.insert(0, REPO_DIR)
print("Installed.")

In [ ]:
# @title Hugging Face login
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if not token:
    from getpass import getpass
    token = getpass("No HF_TOKEN secret found. Paste a Hugging Face read token: ")
login(token=token, add_to_git_credential=False)
del token
print("Logged in. If you haven't yet, accept the model licenses here:")
print("  https://huggingface.co/bodhan-ai/indic-transcribe-flex")
print("  https://huggingface.co/bodhan-ai/indic-transcribe-core")

In [ ]:
# @title Launch (keeps running; the link appears below)
from transcriber.launch import serve
serve(engine=DEFAULT_ENGINE)